# 03 - Data Cleaning
Load the raw 2023 F1 race data and clean it for ML.
- Handle missing values
- Fix data types
- Remove unusable rows
- Save cleaned data to /data folder

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
df = pd.read_csv('../data/f1_2023_all_races.csv')
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()

Shape: (440, 13)

Columns: ['DriverNumber', 'BroadcastName', 'FullName', 'TeamName', 'Position', 'Points', 'Status', 'GridPosition', 'Time', 'Year', 'RaceName', 'Round', 'Top3']


,DriverNumber,BroadcastName,FullName,TeamName,Position,Points,Status,GridPosition,Time,Year,RaceName,Round,Top3
0,1,M VERSTAPPEN,Max Verstappen,Red Bull Racing,1.0,25.0,Finished,1.0,0 days 01:33:56.736000,2023,Bahrain,1,1
1,11,S PEREZ,Sergio Perez,Red Bull Racing,2.0,18.0,Finished,2.0,0 days 00:00:11.987000,2023,Bahrain,1,1
2,14,F ALONSO,Fernando Alonso,Aston Martin,3.0,15.0,Finished,5.0,0 days 00:00:38.637000,2023,Bahrain,1,1
3,55,C SAINZ,Carlos Sainz,Ferrari,4.0,12.0,Finished,4.0,0 days 00:00:48.052000,2023,Bahrain,1,0
4,44,L HAMILTON,Lewis Hamilton,Mercedes,5.0,10.0,Finished,7.0,0 days 00:00:50.977000,2023,Bahrain,1,0


In [3]:
print("Missing values in each column:")
print(df.isnull().sum())
print("\nTotal missing:", df.isnull().sum().sum())

Missing values in each column:
DriverNumber      0
BroadcastName     1
FullName          0
TeamName          0
Position          1
Points            0
Status            0
GridPosition      1
Time             64
Year              0
RaceName          0
Round             0
Top3              0
dtype: int64

Total missing: 67


In [4]:
print("Data types:")
print(df.dtypes)

Data types:
DriverNumber       int64
BroadcastName     object
FullName          object
TeamName          object
Position         float64
Points           float64
Status            object
GridPosition     float64
Time              object
Year               int64
RaceName          object
Round              int64
Top3               int64
dtype: object


In [5]:
print("Unique teams:", df['TeamName'].unique())
print("\nUnique statuses:", df['Status'].unique())
print("\nUnique positions:", sorted(df['Position'].dropna().unique()))

Unique teams: ['Red Bull Racing' 'Aston Martin' 'Ferrari' 'Mercedes' 'Alfa Romeo'
 'Alpine' 'Williams' 'AlphaTauri' 'Haas F1 Team' 'McLaren']

Unique statuses: ['Finished' 'Lapped' 'Retired' 'Accident' 'Undertray' 'Did not start'
 'Withdrew' 'Disqualified' 'Collision damage']

Unique positions: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(18.0), np.float64(19.0), np.float64(20.0)]


In [11]:
# Fill missing positions with 20 (last place) then convert to int
df['Position'] = df['Position'].fillna(20).astype(int)

print("Position column fixed!")
print(df['Position'].value_counts().sort_index())

Position column fixed!
Position
1     22
2     22
3     22
4     22
5     22
6     22
7     22
8     22
9     22
10    22
11    22
12    22
13    22
14    22
15    22
16    22
17    22
18    22
19    22
20    22
Name: count, dtype: int64


In [12]:
# Some drivers start from pit lane — give them grid position 20
df['GridPosition'] = df['GridPosition'].fillna(20).astype(int)

print("GridPosition column fixed!")
print(df['GridPosition'].describe())

GridPosition column fixed!
count    440.000000
mean      10.409091
std        5.780015
min        0.000000
25%        5.000000
50%       10.000000
75%       15.000000
max       20.000000
Name: GridPosition, dtype: float64


In [13]:
# If status is 'Finished' or contains '+' (like '+1 Lap') = finished the race
df['Finished'] = df['Status'].apply(
    lambda x: 1 if ('Finished' in str(x) or '+' in str(x)) else 0
)

print("Finished column created!")
print(df['Finished'].value_counts())

Finished column created!
Finished
1    304
0    136
Name: count, dtype: int64


In [14]:
df['Points'] = df['Points'].fillna(0).astype(float)

print("Points column fixed!")
print(df['Points'].describe())

Points column fixed!
count    440.000000
mean       5.095455
std        7.258644
min        0.000000
25%        0.000000
50%        0.500000
75%        9.250000
max       26.000000
Name: Points, dtype: float64


In [15]:
df = df.drop(columns=['Time'], errors='ignore')

print("Time column dropped!")
print("Remaining columns:", df.columns.tolist())

Time column dropped!
Remaining columns: ['DriverNumber', 'BroadcastName', 'FullName', 'TeamName', 'Position', 'Points', 'Status', 'GridPosition', 'Year', 'RaceName', 'Round', 'Top3', 'Finished']


In [16]:
# Get all unique teams and assign each a number
team_mapping = {team: idx for idx, team in enumerate(df['TeamName'].unique())}
df['TeamID'] = df['TeamName'].map(team_mapping)

print("Team mapping:")
for team, idx in team_mapping.items():
    print(f"  {team} → {idx}")

Team mapping:
  Red Bull Racing → 0
  Aston Martin → 1
  Ferrari → 2
  Mercedes → 3
  Alfa Romeo → 4
  Alpine → 5
  Williams → 6
  AlphaTauri → 7
  Haas F1 Team → 8
  McLaren → 9


In [17]:
df['DriverNumber'] = df['DriverNumber'].astype(int)
print("DriverNumber fixed!")


DriverNumber fixed!


In [18]:
print("Final shape:", df.shape)
print("\nMissing values remaining:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)
df.head(50)

Final shape: (440, 14)

Missing values remaining:
DriverNumber     0
BroadcastName    1
FullName         0
TeamName         0
Position         0
Points           0
Status           0
GridPosition     0
Year             0
RaceName         0
Round            0
Top3             0
Finished         0
TeamID           0
dtype: int64

Data types:
DriverNumber       int64
BroadcastName     object
FullName          object
TeamName          object
Position           int64
Points           float64
Status            object
GridPosition       int64
Year               int64
RaceName          object
Round              int64
Top3               int64
Finished           int64
TeamID             int64
dtype: object


,DriverNumber,BroadcastName,FullName,TeamName,Position,Points,Status,GridPosition,Year,RaceName,Round,Top3,Finished,TeamID
0,1,M VERSTAPPEN,Max Verstappen,Red Bull Racing,1,25.0,Finished,1,2023,Bahrain,1,1,1,0
1,11,S PEREZ,Sergio Perez,Red Bull Racing,2,18.0,Finished,2,2023,Bahrain,1,1,1,0
2,14,F ALONSO,Fernando Alonso,Aston Martin,3,15.0,Finished,5,2023,Bahrain,1,1,1,1
3,55,C SAINZ,Carlos Sainz,Ferrari,4,12.0,Finished,4,2023,Bahrain,1,0,1,2
4,44,L HAMILTON,Lewis Hamilton,Mercedes,5,10.0,Finished,7,2023,Bahrain,1,0,1,3
5,18,L STROLL,Lance Stroll,Aston Martin,6,8.0,Finished,8,2023,Bahrain,1,0,1,1
6,63,G RUSSELL,George Russell,Mercedes,7,6.0,Finished,6,2023,Bahrain,1,0,1,3
7,77,V BOTTAS,Valtteri Bottas,Alfa Romeo,8,4.0,Finished,12,2023,Bahrain,1,0,1,4
8,10,P GASLY,Pierre Gasly,Alpine,9,2.0,Finished,20,2023,Bahrain,1,0,1,5
9,23,A ALBON,Alexander Albon,Williams,10,1.0,Finished,15,2023,Bahrain,1,0,1,6


In [20]:
print("Top3 distribution:")
print(df['Top3'].value_counts())
print(f"\n{df['Top3'].sum()} top-3 finishes out of {len(df)} total entries")

Top3 distribution:
Top3
0    374
1     66
Name: count, dtype: int64

66 top-3 finishes out of 440 total entries


In [21]:
df.to_csv('../data/f1_2023_cleaned.csv', index=False)
print(f"Cleaned data saved! Shape: {df.shape} ✅")

Cleaned data saved! Shape: (440, 14) ✅


## Cleaning Summary
- Loaded 440 rows of raw F1 2023 race data
- Fixed Position and GridPosition columns (float → int)
- Created Finished column from Status
- Fixed Points column
- Dropped Time column (not useful for ML)
- Encoded TeamName as numbers (TeamID)
- Zero missing values remaining
- Saved to /data/f1_2023_cleaned.csv